In [0]:
# window functions

In [0]:
cus_df = spark.table("samples.bakehouse.sales_customers")
cus_df.display()

In [0]:
sales_df = spark.table("samples.bakehouse.sales_transactions")
sales_df.display()

In [0]:
sales_df.createOrReplaceTempView("sales")

In [0]:
%sql
select * from sales;

In [0]:
%sql
-- Find the custormer who did the second highest transaction

select 
*
from sales
qualify dense_rank() over (order by totalPrice desc) = 2;


In [0]:
%sql

select 
customerID,
count(*) as dup_cnt
from sales
group by customerID
having count(*) > 1;

In [0]:
%sql

with combined_customer_transaction_value as
(
    select
    customerID,
    sum(totalPrice) as total_transaction_value
    from sales
    group by customerID
)
select * from combined_customer_transaction_value;


In [0]:
%sql

with combined_customer_transaction_value as
(
    select
    customerID,
    sum(totalPrice) as total_transaction_value
    from sales
    group by customerID
)
select * from combined_customer_transaction_value
qualify dense_rank() over(order by total_transaction_value desc) = 2;


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
sales_df = (
    sales_df
    .groupBy("customerID")
    .agg(
        F.sum("totalPrice").alias("total_transaction_value")
    )
)

sales_df.display()

In [0]:
w = Window.orderBy(F.col("total_transaction_value").desc())

sales_df = (
    sales_df
    .withColumn("rk", F.dense_rank().over(w))
    .filter(F.col("rk") == 2)
    .drop("rk")
)

sales_df.display()

In [0]:
%sql
-- Find out the third highest transaction value from each franchiseID in the sales table

select
*
from sales
qualify dense_rank() over(partition by franchiseID order by totalPrice desc) = 3;

In [0]:
w = Window.partitionBy("franchiseID").orderBy(F.col("totalPrice").desc())

new_sales_df = (
    sales_df
    .withColumn("rk", F.dense_rank().over(w))
    .filter(F.col("rk") == 3)
    .drop("rk")
)

display(new_sales_df)

In [0]:
# row_number, rank, dense_rank, lead, lag
# avg, count, sum, min, max

In [0]:
# Find the employees who earn more then their dept avg salary